## __LSEG Scope 2 Data Extraction - MASTER__

__Noah Proctor__

The purpose of this notebook is to test the algorithm I employ, so I explain this program for my purpose, and thus, will leave out details I feel I don;t need to be reminded of. There are far too many companies we wish to extract data from to reasonably test the program simultaneously. I focus on companies with known ESG data for extraction (Microsoft, Apple, Google) to make sure all aspects of the program function properly. Because this is a test notebook, I will explain each step I took.

Broadly, this data is purely for extraction purposes, and will be only used for further data analysis and reconciliation. Thus, my goal was to keep this as simple of a process as possible.

__Step 1: Open LSEG__

In [2]:
import lseg.data as ld
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Imports successful!")
print(f"LSEG Data Library version: {ld.__version__}")

try:
    ld.open_session()
    print("Using active LSEG Workspace session.")
except Exception as e:
    print(f"Error opening session: {e}")

Imports successful!
LSEG Data Library version: 2.1.1
Using active LSEG Workspace session.


__Step 2: Create Ticker CSV__

It's important for us to keep an updated list of all of the tickers on the LSEG due to its updates. Having a list of tickers that does not update automatically could lead to faulty collection, and thus, a faulty analysis. We retrieve the data using an LSEG API and select the columns to export to the CSV.

In [3]:
# ----- initialize dataframe -----
df = ld.get_data(["SCREEN(U(IN(Equity(primary))), TR.HasESGCoverage==true, IN(TR.HeadquartersRegion,""Americas"",), CURN=USD)"],["TR.CommonName;TR.HasESGCoverage;TR.HeadquartersRegion;TR.ISIN;TR.IssuerTickerCode"])

# ----- export a csv of companies and rics -----
output_filename = 'lseg_tickers.csv'
exclude = 'Masterworks'     # filter all of the Masterworks CCNs
filtered_df = df[~df['Company Common Name'].str.contains(exclude, case=False, na=False)]

try:
    filtered_df.to_csv(output_filename, columns=['Company Common Name', 'Instrument', 'Issuer Ticker'], index=False)
    print(f"Successfully exported columns RIC data to {output_filename}")
except Exception as e:
    print(f"Failed to export to CSV: {e}")

# display this particular dataframe
df

Successfully exported columns RIC data to lseg_tickers.csv


,Instrument,Company Common Name,ESG Coverage Flag,Region of Headquarters,ISIN,Issuer Ticker
0,ECL.N,Ecolab Inc,True,Americas,US2788651006,ECL
1,EEI.OQ^A20,Ecology and Environment Inc,True,Americas,US2788781035,WSPXEC
2,EVT.TO,Economic Investment Trust Ltd,True,Americas,CA2788931020,EVT
3,EDUC.OQ,Educational Development Corp,True,Americas,US2814791057,EDUC
4,STZ.N,Constellation Brands Inc,True,Americas,US21036P1084,STZ
...,...,...,...,...,...,...
12785,SWDR.PK,Starwood Real Estate Income Trust Inc,True,Americas,US85570X4051,SWDR
12786,SURF_u.V,Starlight US Residential Multi-Family Investme...,True,Americas,CA85555K2048,
12787,AUAU3.SA,Uniao Pet Participacoes SA,True,Americas,BRAUAUACNOR1,AUAUC
12788,ALBC.PK,Alternative Ballistics Corp,True,Americas,US02158B1017,


In [22]:
# ---------- scrape compustat data and combine with lseg tickers ----------
import pandas as pd
import re
from thefuzz import process, fuzz
from datetime import datetime

LSEG_FILE = 'lseg_tickers.csv'
COMPUSTAT_FILE = 'Compustat_Identifiers.csv'
OUTPUT_FILE = 'lseg_with_compustat.csv'

def clean_company_name(name):
    """
    Remove common suffixes and standardize company names.
    """
    if pd.isna(name):
        return ""
    
    name = str(name).upper().strip()
    
    # remove these words
    words_to_remove = [
        'INCORPORATED', 'INCORPORATION', 'CORPORATION', 'CORP',
        'COMPANY', 'CO', 'INC', 'LLC', 'LTD', 'LIMITED', 'PLC',
        'HOLDINGS', 'GROUP', 'INTERNATIONAL', 'GLOBAL', 'THE'
    ]
    
    for word in words_to_remove:
        name = re.sub(r'\b' + word + r'\b', '', name)
    
    # remove punctuation
    name = re.sub(r'[.,&\-\'\"()]', ' ', name)
    
    # remove extra spaces
    name = re.sub(r'\s+', ' ', name).strip()
    
    return name

# ---------- LOAD AND PREPARE DATA ----------

print("-"*70)
print("LSEG-COMPUSTAT COMPANY MATCHING WITH FUZZY LOGIC")
print("-"*70)

print("\nLOADING DATA...")
lseg = pd.read_csv(LSEG_FILE)
compustat = pd.read_csv(COMPUSTAT_FILE)

print(f"  LSEG: {len(lseg):,} companies")
print(f"  Compustat: {len(compustat):,} records")

# clean names
print("\nCLEANING COMPANY NAMES...")
lseg['Clean_Name'] = lseg['Company Common Name'].apply(clean_company_name)
compustat['Clean_Name'] = compustat['conm'].apply(clean_company_name)

# get unique Compustat companies (most recent record)
print("GETTING UNIQUE COMPUSTAT COMPANIES...")
compustat['datadate'] = pd.to_datetime(compustat['datadate'])
compustat_unique = compustat.sort_values('datadate').groupby('gvkey').last().reset_index()
print(f"  Unique: {len(compustat_unique):,} companies")

# ---------- STEP 1: EXACT MATCHING ----------

print("\n" + "-"*70)
print("STEP 1: EXACT MATCHING ON CLEANED NAMES")
print("-"*70)

# merge on cleaned names
result = lseg.merge(
    compustat_unique[['Clean_Name', 'gvkey', 'conm']],
    on='Clean_Name',
    how='left'
)

# rename columns
result = result.rename(columns={
    'gvkey': 'GVKEY',
    'conm': 'Compustat_Company_Name'
})

exact_matched = result['GVKEY'].notna().sum()
exact_unmatched = result['GVKEY'].isna().sum()

print(f"✓ Exact matches: {exact_matched:,} ({exact_matched/len(lseg)*100:.1f}%)")
print(f"⚠ Unmatched: {exact_unmatched:,} ({exact_unmatched/len(lseg)*100:.1f}%)")

# ---------- STEP 2: FUZZY MATCHING FOR UNMATCHED COMPANIES ---------

print("\n" + "-"*70)
print("STEP 2: FUZZY MATCHING FOR UNMATCHED COMPANIES")
print("-"*70)

# get unmatched companies
unmatched_lseg = result[result['GVKEY'].isna()].copy()
print(f"Companies to fuzzy match: {len(unmatched_lseg):,}")

if len(unmatched_lseg) > 0:
    # handle duplicate cleaned names by keeping most recent
    compustat_for_fuzzy = compustat_unique.drop_duplicates(subset='Clean_Name', keep='last')
    
    print(f"Compustat companies after deduplication: {len(compustat_for_fuzzy):,}")
    
    # prepare Compustat lookup
    compustat_names = compustat_for_fuzzy['Clean_Name'].values
    compustat_lookup = compustat_for_fuzzy.set_index('Clean_Name')[['gvkey', 'conm']].to_dict('index')
    
    # fuzzy matching settings
    FUZZY_THRESHOLD = 80  # Lower this if you want more matches (e.g., 70, 75)
    
    print(f"Fuzzy matching threshold: {FUZZY_THRESHOLD}")
    print("Processing fuzzy matches...")
    
    fuzzy_matches = []
    fuzzy_count = 0
    
    for idx, row in unmatched_lseg.iterrows():
        lseg_name = row['Clean_Name']
        
        if not lseg_name or lseg_name == "":
            continue
        
        # find best match using token_sort_ratio
        match_result = process.extractOne(
            lseg_name, 
            compustat_names,
            scorer=fuzz.token_sort_ratio
        )
        
        if match_result and match_result[1] >= FUZZY_THRESHOLD:
            matched_name = match_result[0]
            score = match_result[1]
            
            # Get GVKEY and company name
            compustat_data = compustat_lookup[matched_name]
            
            fuzzy_matches.append({
                'index': idx,
                'lseg_name': row['Company Common Name'],
                'lseg_clean': lseg_name,
                'compustat_name': compustat_data['conm'],
                'compustat_clean': matched_name,
                'gvkey': compustat_data['gvkey'],
                'score': score
            })
            fuzzy_count += 1
            
            # show examples of fuzzy matches (first 10)
            if fuzzy_count <= 10:
                print(f"  LSEG: '{row['Company Common Name'][:50]}'")
                print(f"  → Compustat: '{compustat_data['conm'][:50]}' (score: {score})")
                print()
    
    print(f"✓ Fuzzy matches found: {fuzzy_count:,}")
    
    # apply fuzzy matches to result
    for match in fuzzy_matches:
        result.loc[match['index'], 'GVKEY'] = match['gvkey']
        result.loc[match['index'], 'Compustat_Company_Name'] = match['compustat_name']
    
    # save fuzzy match details for review
    if fuzzy_matches:
        fuzzy_df = pd.DataFrame(fuzzy_matches)
        fuzzy_df.to_csv('fuzzy_matches_review.csv', index=False)
        print(f"✓ Saved fuzzy matches to: fuzzy_matches_review.csv")

# FINAL RESULTS

print("\n" + "="*70)
print("FINAL MATCHING RESULTS")
print("="*70)

final_matched = result['GVKEY'].notna().sum()
final_unmatched = result['GVKEY'].isna().sum()

print(f"\nTotal companies: {len(lseg):,}")
print(f"  ✓ Matched (exact): {exact_matched:,} ({exact_matched/len(lseg)*100:.1f}%)")
print(f"  ✓ Matched (fuzzy): {fuzzy_count:,} ({fuzzy_count/len(lseg)*100:.1f}%)")
print(f"  ✓ TOTAL MATCHED: {final_matched:,} ({final_matched/len(lseg)*100:.1f}%)")
print(f"  ✗ Still unmatched: {final_unmatched:,} ({final_unmatched/len(lseg)*100:.1f}%)")

# Remove the temporary Clean_Name column
result = result.drop(columns=['Clean_Name'])

# ---------- SAVE OUTPUTS ----------

print("\n" + "-"*70)
print("SAVING OUTPUT FILES")
print("-"*70)

# save complete results
result.to_csv(OUTPUT_FILE, index=False)
print(f"✓ SAVED: {OUTPUT_FILE}")

# save unmatched companies if any
if final_unmatched > 0:
    print(f"\n⚠ WARNING: {final_unmatched:,} companies still not matched!")
    print("\nFirst 20 unmatched companies:")
    unmatched_sample = result[result['GVKEY'].isna()][['Company Common Name', 'Instrument']].head(20)
    print(unmatched_sample.to_string(index=False))
    
    # save all unmatched for review
    result[result['GVKEY'].isna()].to_csv('unmatched_companies.csv', index=False)
    print(f"\n✓ Saved unmatched companies to: unmatched_companies.csv")

print("\n" + "="*70)
print("MATCHING COMPLETE!")
print("="*70)
print("\nOutput files:")
print("  1. lseg_with_compustat.csv - All companies (matched + unmatched)")
print("  2. fuzzy_matches_review.csv - Details of fuzzy matches for verification")
print("  3. unmatched_companies.csv - Companies still needing manual review")

----------------------------------------------------------------------
LSEG-COMPUSTAT COMPANY MATCHING WITH FUZZY LOGIC
----------------------------------------------------------------------

LOADING DATA...
  LSEG: 12,610 companies
  Compustat: 185,627 records

CLEANING COMPANY NAMES...
GETTING UNIQUE COMPUSTAT COMPANIES...
  Unique: 24,487 companies

----------------------------------------------------------------------
STEP 1: EXACT MATCHING ON CLEANED NAMES
----------------------------------------------------------------------
✓ Exact matches: 6,970 (55.3%)
⚠ Unmatched: 5,706 (45.2%)

STEP 2: FUZZY MATCHING FOR UNMATCHED COMPANIES
Companies to fuzzy match: 5,706
Compustat companies after deduplication: 24,430
Fuzzy matching threshold: 80
Processing fuzzy matches...
  LSEG: 'Economic Investment Trust Ltd'
  → Compustat: 'ECONOMIC INVESTMENT TR LTD' (score: 94)

  LSEG: 'Electro Scientific Industries Inc'
  → Compustat: 'ELECTRO SCIENTIFIC INDS INC' (score: 88)

  LSEG: 'Electronic T

__Step 3: Set Conditions for Collection__:

Read in the lseg_tickers.csv and convert to a list. All other boundary conditions are set. Notably, in an environment with a large amount of fields to extract, data may be thrown out of the stack in order of data, so that the most recent data will come first. In that case, we just decrease the volume of the list that we run at a time, and create multiple different CSVs.

In [23]:
# ----- intialize conditions -----
rics_df = pd.read_csv('lseg_with_compustat.csv')       # reads the RICs in the dataframe, uses for data extraction
COMPANY_RICS = rics_df['Instrument'].tolist()

# date range
START_DATE = '2010-01-01'
END_DATE = '2024-12-31'

# batch size (process companies in batches)
BATCH_SIZE = 50

print(f"Companies to extract: {len(COMPANY_RICS)}")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Batch size: {BATCH_SIZE}")

Companies to extract: 12676
Date range: 2010-01-01 to 2024-12-31
Batch size: 50


__Step 4: Determine Fields to Extract__

Straightforward, but add more fields to the actual program.

In [24]:
# adjust these fields as needed

ESG_FIELDS = [
    # basic company info
    'TR.CommonName',                                                # company name
    'TR.TRBCIndustryGroup',                                         # Industry classification
    'TR.EventStartDate'
    
    # scope 2 emissions direct
    'TR.Scope2EstTotal',                                            # total Scope 2
    'TR.CDPCO2EquivalentEmissionIndirectScope2Locationbased',       # location-based
    'TR.CDPCO2EquivalentEmissionIndirectScope2Marketbased',         # market-based
    
    # additional Scope data
    'TR.Scope1EstTotal',                                            # Scope 1 emissions
    'TR.Scope3EstTotal',                                            # Scope 3 emissions
    'TR.CO2EquivalentEmissionTotal',                                # total GHG emissions
    'TR.CDPCO2DirectScope1',                                        # direct Scope 1
    
    # total emissions (location/market based)
    'TR.CO2EquivalentEmissionTotalMarketbased',                     # total market-based
    'TR.CO2EquivalentEmissionTotalLocationbased',                   # total location-based
    'TR.CO2EmissionTotal',                                          # total CO2
    
    # emissions intensity
    'TR.GHGEmissionsIntensityScope1and2and3ParisAligned',           # intensity metric
    
    # energy consumption
    'TR.EnergyUseTotal',                                            # total energy
    'TR.TotalRenewableEnergy',                                      # renewable energy
    
    # ESG scores
    'TR.TRESGScoreGrade',                                           # ESG score grade
    'TR.FundRIESGEnviron',                                          # environmental score
]

__Step 5: Collect Data with Batch Processing__

Process all companies from lseg_tickers.csv in batches to avoid overwhelming the API. Progress is tracked in real-time, and errors are logged for review. Each batch processes BATCH_SIZE companies (default: 50) with a 1-second pause between batches.

In [25]:
import time
from datetime import datetime
import pickle
import os

# initialize tracking
all_data = []
failed_batches = []
total_batches = (len(COMPANY_RICS) + BATCH_SIZE - 1) // BATCH_SIZE

# checkpoint file for recovery
checkpoint_file = 'extraction_checkpoint.pkl'

# try to load from checkpoint if it exists
if os.path.exists(checkpoint_file):
    print("Found checkpoint file. Loading progress...")
    with open(checkpoint_file, 'rb') as f:
        checkpoint = pickle.load(f)
        all_data = checkpoint['all_data']
        failed_batches = checkpoint['failed_batches']
        start_batch = checkpoint['last_batch'] + 1
    print(f"Resuming from batch {start_batch}/{total_batches}\n")
else:
    start_batch = 0

print(f"Starting batch extraction...")
print(f"Total companies: {len(COMPANY_RICS)}")
print(f"Total batches: {total_batches}")
print(f"Batch size: {BATCH_SIZE}")
print("-" * 70)

# process in batches
for batch_num in range(start_batch, total_batches):
    start_idx = batch_num * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(COMPANY_RICS))
    batch_rics = COMPANY_RICS[start_idx:end_idx]
    
    print(f"\n[Batch {batch_num + 1}/{total_batches}] Processing {len(batch_rics)} companies...")
    print(f"  Range: indices {start_idx} to {end_idx - 1}")
    print(f"  Time: {datetime.now().strftime('%H:%M:%S')}")
    
    try:
        # extract data for this batch
        response = ld.get_history(
            universe=batch_rics,
            fields=ESG_FIELDS,
            interval="1Y",
            start=START_DATE,
            end=END_DATE
        )
        
        # add to collection
        all_data.append(response)
        print(f"  ✓ Success! Shape: {response.shape}")
        
        # save checkpoint every 10 batches
        if (batch_num + 1) % 10 == 0:
            with open(checkpoint_file, 'wb') as f:
                pickle.dump({
                    'all_data': all_data,
                    'failed_batches': failed_batches,
                    'last_batch': batch_num
                }, f)
            print(f"  💾 Checkpoint saved")
        
        # brief pause between batches
        if batch_num < total_batches - 1:
            time.sleep(1)
            
    except Exception as e:
        print(f"  ✗ ERROR in batch {batch_num + 1}: {str(e)[:100]}")
        failed_batches.append({
            'batch_num': batch_num + 1,
            'start_idx': start_idx,
            'end_idx': end_idx,
            'rics': batch_rics,
            'error': str(e)
        })
        continue

# final checkpoint save
with open(checkpoint_file, 'wb') as f:
    pickle.dump({
        'all_data': all_data,
        'failed_batches': failed_batches,
        'last_batch': total_batches - 1
    }, f)

print("\n" + "=" * 70)
print(f"EXTRACTION COMPLETE")
print(f"Successful batches: {len(all_data)}/{total_batches}")
print(f"Failed batches: {len(failed_batches)}")

if failed_batches:
    print("\n⚠ FAILED BATCHES:")
    for fb in failed_batches:
        print(f"  Batch {fb['batch_num']}: indices {fb['start_idx']}-{fb['end_idx']}")
        print(f"    Error: {fb['error'][:100]}...")
else:
    print("\n✓ All batches completed successfully!")

print(f"\n💾 Checkpoint saved to {checkpoint_file}")
print("   (Delete this file when extraction is complete)")

Found checkpoint file. Loading progress...
Resuming from batch 254/254

Starting batch extraction...
Total companies: 12676
Total batches: 254
Batch size: 50
----------------------------------------------------------------------

EXTRACTION COMPLETE
Successful batches: 253/254
Failed batches: 3

⚠ FAILED BATCHES:
  Batch 159: indices 7900-7950
    Error: UDF Core request failed. Service Temporarily Unavailable Requested universes: ['MYCC.K^I17', 'CACQ.O...
  Batch 160: indices 7950-8000
    Error: Backend error. 400 Bad Request Requested universes: ['PINC.OQ^K25', 'SEMI.O^L16', 'FTXP.PK', 'SEER3....
  Batch 181: indices 9000-9050
    Error: Backend error. 400 Bad Request Requested universes: ['BL.OQ', 'RARX.OQ^D20', 'ARCH.N^A25', 'ERE_u.TO...

💾 Checkpoint saved to extraction_checkpoint.pkl
   (Delete this file when extraction is complete)


__Step 5.5: Retry Failed Batches (Optional)__

If any batches failed, run this cell to retry them. You can also adjust the batch to retry specific indices.

In [26]:
# only run this if there were failed batches
if failed_batches:
    print(f"Retrying {len(failed_batches)} failed batches...\n")
    
    retry_success = []
    still_failed = []
    
    for fb in failed_batches:
        print(f"Retrying batch {fb['batch_num']}...")
        
        try:
            response = ld.get_history(
                universe=fb['rics'],
                fields=ESG_FIELDS,
                interval="1Y",
                start=START_DATE,
                end=END_DATE
            )
            
            all_data.append(response)
            retry_success.append(fb['batch_num'])
            print(f"  ✓ Success!")
            time.sleep(1)
            
        except Exception as e:
            print(f"  ✗ Still failed: {str(e)[:100]}")
            still_failed.append(fb)
    
    print(f"\nRetry complete:")
    print(f"  Successful: {len(retry_success)}")
    print(f"  Still failed: {len(still_failed)}")
    
    # Update failed_batches list
    failed_batches = still_failed
else:
    print("No failed batches to retry!")

Session is not opened. Can't send any request
Session is not opened. Can't send any request
Session is not opened. Can't send any request


Retrying 3 failed batches...

Retrying batch 159...
  ✗ Still failed: Session is not opened. Can't send any request
Retrying batch 160...
  ✗ Still failed: Session is not opened. Can't send any request
Retrying batch 181...
  ✗ Still failed: Session is not opened. Can't send any request

Retry complete:
  Successful: 0
  Still failed: 3


__Step 6: Combine and Sort__

This step primes the data to be converted into a CSV while making sure that it is sorted according to data and company. It also keeps track of when I extracted the data, creating a separate spreadsheet each time.

In [27]:
# ----- make sure that the data extraction worked, then combine + reshape -----
if not all_data:
    print("ERROR: No data was extracted!")
else:
    print(f"Reshaping {len(all_data)} batches...")
    
    # fastest method is list comprehension with concatenation
    reshaped_batches = [
        batch.stack(level=0).reset_index().rename(
            columns={'level_1': 'Instrument', 'Date': 'ReportDate'}
        )
        for batch in all_data
    ]
    
    # Combine
    scope2_data = pd.concat(reshaped_batches, ignore_index=True)
    
    print(f"✓ Successfully reshaped {len(scope2_data)} total records")
    
    # Sort and add metadata
    scope2_data = scope2_data.sort_values(['Instrument', 'ReportDate'])
    scope2_data['DataExtractedOn'] = datetime.now()

Reshaping 253 batches...
✓ Successfully reshaped 88745 total records


__Step 6.5: Preview the Reshaped Data__

Let's verify the data is in the correct format before saving.

In [28]:
# show data structure
print(f"Shape: {scope2_data.shape}")
print(f"\nColumns: {list(scope2_data.columns)}")
print(f"\nRecords per instrument:")
print(scope2_data['Instrument'].value_counts().sort_index())
print("\nFirst 10 rows:")
scope2_data.head(10)

Shape: (88745, 15)

Columns: ['ReportDate', 'Instrument', 'Company Common Name', 'TRBC Industry Group Name', 'CDP CO2 Equivalent Emission Indirect Scope 2 Location-based', 'CDP CO2 Equivalent Emission Indirect Scope 2 Market-based', 'Scope 1 Combined (Reported or Estimated) Total', 'Scope 3 Combined (Reported or Estimated) Total', 'CDP CO2 Equivalent Emissions Direct Scope 1', 'CO2 Equivalent Emissions Total', 'Energy Use Total', 'Total Renewable Energy', 'ESG Score Grade', 'ESG-Environmental', 'DataExtractedOn']

Records per instrument:
Instrument
0KIW.L^G20     12
0US7.L         12
0VIO.L^E21     10
1316.HK        15
1521.HK        15
               ..
ZY.OQ^J22      12
ZYME.OQ        15
ZYNE.OQ^J23    15
ZYXIQ.PK       15
ZZZ.TO^J24     14
Name: count, Length: 6018, dtype: int64

First 10 rows:


,ReportDate,Instrument,Company Common Name,TRBC Industry Group Name,CDP CO2 Equivalent Emission Indirect Scope 2 Location-based,CDP CO2 Equivalent Emission Indirect Scope 2 Market-based,Scope 1 Combined (Reported or Estimated) Total,Scope 3 Combined (Reported or Estimated) Total,CDP CO2 Equivalent Emissions Direct Scope 1,CO2 Equivalent Emissions Total,Energy Use Total,Total Renewable Energy,ESG Score Grade,ESG-Environmental,DataExtractedOn
23513,2007-12-31,0KIW.L^G20,,,,,<NA>,<NA>,,,,,,<NA>,2026-03-13 15:03:26.025686
23514,2008-12-31,0KIW.L^G20,,,,,<NA>,<NA>,,,,,,<NA>,2026-03-13 15:03:26.025686
23517,2009-12-31,0KIW.L^G20,,,,,<NA>,<NA>,,,,,,<NA>,2026-03-13 15:03:26.025686
23530,2010-12-31,0KIW.L^G20,,,,,<NA>,<NA>,,,,,,<NA>,2026-03-13 15:03:26.025686
23559,2011-12-31,0KIW.L^G20,,,,,<NA>,<NA>,,,,,,<NA>,2026-03-13 15:03:26.025686
23588,2012-12-31,0KIW.L^G20,,,,,<NA>,<NA>,,,,,,<NA>,2026-03-13 15:03:26.025686
23617,2013-12-31,0KIW.L^G20,,,,,<NA>,<NA>,,,,,,<NA>,2026-03-13 15:03:26.025686
23646,2014-12-31,0KIW.L^G20,,,,,<NA>,<NA>,,,,,,<NA>,2026-03-13 15:03:26.025686
23673,2015-12-31,0KIW.L^G20,,,,,<NA>,<NA>,,,,,,<NA>,2026-03-13 15:03:26.025686
23701,2016-12-31,0KIW.L^G20,,,,,<NA>,<NA>,,,,,,<NA>,2026-03-13 15:03:26.025686


__Step 8: Save to CSV__

Straightforward. Different days of extraction will create different files.

In [29]:
# ---------- merge the compustat data with the extracted esg data ----------
from thefuzz import process

rics_df = rics_df.reset_index(drop=True)
scope2_df = scope2_data.reset_index(drop=True)

# Drop empty columns from scope2
empty_cols = ['Company Common Name', 'TRBC Industry Group Name', 'ESG-Environmental']
scope2_data = scope2_data.drop(columns=[col for col in empty_cols if col in scope2_df.columns])

# Prepare mapping (one row per Instrument)
mapping = rics_df[['Instrument', 'GVKEY', 'Compustat_Company_Name']].drop_duplicates('Instrument')

# Merge (this adds GVKEY and company name to EVERY row in scope2)
result = scope2_data.merge(mapping, on='Instrument', how='left')

# Check results
print(f"✓ Total records: {len(result)}")
print(f"✓ Records with GVKEY: {result['GVKEY'].notna().sum()}")

csv_filename = f'scope2_esg_data_{datetime.now().strftime("%Y%m%d")}.csv'   # keeps track of when extraction occurs
result.to_csv(csv_filename, index=False)
print(f"Data saved to: {csv_filename}")
print(f"\nTotal records saved: {len(scope2_data):,}")

✓ Total records: 88745
✓ Records with GVKEY: 77834
Data saved to: scope2_esg_data_20260313.csv

Total records saved: 88,745


__Step 9: Close the Session__

In [30]:
try:
    ld.close_session()
    print("Session closed")
except:
    pass

Session closed
